# Step 4: A/B Test Statistical Inference Engine

**Objective:** Rigorously evaluate the outcome of our product variant experiment using inferential statistics.

**Experiment Specs:**
* **Control Group:** Baseline user experience.
* **Treatment Group:** Modified product variant (e.g., streamlined checkout / updated UX).
* **Primary Metric:** Binary Conversion Rate (`converted` = 1 if user made a purchase, 0 otherwise).
* **Null Hypothesis ($H_0$):** $p_{\text{treatment}} = p_{\text{control}}$ (No difference in conversion rates).
* **Alternative Hypothesis ($H_1$):** $p_{\text{treatment}} \neq p_{\text{control}}$ (Statistically significant difference exists).
* **Significance Threshold ($\alpha$):** 0.05

In [5]:
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from sqlalchemy import create_engine
from dotenv import load_dotenv
from urllib.parse import quote_plus

# Database Connection Setup
load_dotenv(override=True)

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', '127.0.0.1').strip()
DB_PORT = os.getenv('DB_PORT', '3306').strip()
DB_NAME = os.getenv('DB_NAME')

encoded_password = quote_plus(DB_PASSWORD)
connection_string = f"mysql+pymysql://{DB_USER}:{encoded_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string, pool_pre_ping=True)

# Fetch A/B Test Data from MySQL
ab_data = pd.read_sql("SELECT user_id, group_assignment, converted FROM ab_test_assignment;", con=engine)
print(f"✅ Loaded {len(ab_data):,} user experiment records from MySQL.")
ab_data.head()

✅ Loaded 96,096 user experiment records from MySQL.


,user_id,group_assignment,converted
0,861eff4711a542e4b93843c6dd7febb0,treatment,1
1,290c77bc529b7ac935b93aa66c333dc3,control,0
2,060e732b5b29e8181a18229c7b0b2b5e,control,1
3,259dac757896d24d7702b9acbbff3f3c,control,0
4,345ecd01c38d18a9036ed96c73b8d066,treatment,0


## 1. Summary Statistics & Contingency Table
Let's build a cross-tabulation table to inspect total users, conversions, and observed conversion rates per variant.

In [6]:
# Create Contingency Table
contingency_table = pd.crosstab(ab_data['group_assignment'], ab_data['converted'])
contingency_table.columns = ['Not Converted', 'Converted']

# Calculate Metrics
summary_df = ab_data.groupby('group_assignment')['converted'].agg(['count', 'sum', 'mean'])
summary_df.columns = ['Total Users', 'Conversions', 'Conversion Rate']
summary_df['Conversion Rate Pct'] = summary_df['Conversion Rate'].apply(lambda x: f"{x * 100:.2f}%")

print("--- Contingency Table ---")
print(contingency_table)
print("\n--- Summary Statistics ---")
summary_df

--- Contingency Table ---
                  Not Converted  Converted
group_assignment                          
control                   43801       4208
treatment                 43266       4821

--- Summary Statistics ---


,Total Users,Conversions,Conversion Rate,Conversion Rate Pct
group_assignment,,,,
control,48009,4208,0.087650,8.77%
treatment,48087,4821,0.100256,10.03%


## 2. Inferential Hypothesis Testing ($\chi^2$ & Z-Test)
We run both **Pearson's Chi-Square Test** and a **Two-Proportion Z-Test** to derive the $p$-value and determine if we can reject $H_0$.

In [7]:
# 1. Pearson's Chi-Square Test
chi2_stat, p_val_chi2, dof, expected = stats.chi2_contingency(contingency_table)

# 2. Two-Proportion Z-Test
count = summary_df['Conversions'].values
nobs = summary_df['Total Users'].values
z_stat, p_val_z = proportions_ztest(count, nobs)

# 3. 95% Confidence Intervals
(control_ci_low, treatment_ci_low), (control_ci_upp, treatment_ci_upp) = proportion_confint(count, nobs, alpha=0.05)

print("="*50)
print("             STATISTICAL TEST RESULTS            ")
print("="*50)
print(f"Chi-Square Statistic ($\chi^2$): {chi2_stat:.4f}")
print(f"Chi-Square p-value:          {p_val_chi2:.4e}")
print(f"Z-Score Statistic:           {z_stat:.4f}")
print(f"Z-Test p-value:              {p_val_z:.4e}")
print("-"*50)
print(f"Control 95% CI:   [{control_ci_low:.4f}, {control_ci_upp:.4f}]")
print(f"Treatment 95% CI: [{treatment_ci_low:.4f}, {treatment_ci_upp:.4f}]")
print("="*50)

# Final Decision Logic
ALPHA = 0.05
if p_val_chi2 < ALPHA:
    print("🚀 CONCLUSION: Reject the Null Hypothesis (H0)!")
    print("The variant treatment produced a STATISTICALLY SIGNIFICANT lift in user conversion.")
else:
    print("⚠️ CONCLUSION: Fail to Reject the Null Hypothesis (H0).")
    print("There is no statistically significant difference between variants.")

             STATISTICAL TEST RESULTS            
Chi-Square Statistic ($\chi^2$): 44.6942
Chi-Square p-value:          2.3034e-11
Z-Score Statistic:           -6.6964
Z-Test p-value:              2.1358e-11
--------------------------------------------------
Control 95% CI:   [0.0851, 0.0902]
Treatment 95% CI: [0.0976, 0.1029]
🚀 CONCLUSION: Reject the Null Hypothesis (H0)!
The variant treatment produced a STATISTICALLY SIGNIFICANT lift in user conversion.


<>:15: SyntaxWarning: invalid escape sequence '\c'
<>:15: SyntaxWarning: invalid escape sequence '\c'
C:\Users\hp\AppData\Local\Temp\ipykernel_15148\3635597607.py:15: SyntaxWarning: invalid escape sequence '\c'
  print(f"Chi-Square Statistic ($\chi^2$): {chi2_stat:.4f}")


## 3. Business Impact Assessment
Quantifying the lift in absolute and relative terms for executive reporting.

In [8]:
p_control = summary_df.loc['control', 'Conversion Rate']
p_treatment = summary_df.loc['treatment', 'Conversion Rate']

abs_lift = p_treatment - p_control
rel_lift = (p_treatment - p_control) / p_control

print("--- Business Impact ---")
print(f"Absolute Conversion Rate Lift:  {abs_lift * 100:+.2f} percentage points")
print(f"Relative Conversion Rate Lift:  {rel_lift * 100:+.2f}%")

print("\n--- Executive Summary Takeaway ---")
print(f"Rolling out the Treatment variant to 100% of the user base is statistically backed to increase overall conversion rate from {p_control*100:.2f}% to {p_treatment*100:.2f}%.")

--- Business Impact ---
Absolute Conversion Rate Lift:  +1.26 percentage points
Relative Conversion Rate Lift:  +14.38%

--- Executive Summary Takeaway ---
Rolling out the Treatment variant to 100% of the user base is statistically backed to increase overall conversion rate from 8.77% to 10.03%.
